# Module 01: Orders EDA & Cleaning

### 1. Load Raw Data (Bronze Layer)

In [25]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "olist_orders_dataset.csv"

# Ensure the file exists before loading
assert RAW_DATA_PATH.exists(), f"Source file not found at: {RAW_DATA_PATH.resolve()}"

orders = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset shape: {orders.shape}")
orders.head(3)

Loaded raw dataset shape: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


### 2. Data Audit & Missing Values Check
Analyzing data types, missing value distribution, and column cardinality.

In [26]:
audit_df = pd.DataFrame(
    {
        "Data Type": orders.dtypes,
        "Missing Values": orders.isna().sum(),
        "Missing %": (orders.isna().mean() * 100).round(2),
        "Unique Values": orders.nunique(),
    }
)

display(audit_df)

,Data Type,Missing Values,Missing %,Unique Values
order_id,object,0,0.00,99441
customer_id,object,0,0.00,99441
order_status,object,0,0.00,8
order_purchase_timestamp,object,0,0.00,98875
order_approved_at,object,160,0.16,90733
order_delivered_carrier_date,object,1783,1.79,81018
order_delivered_customer_date,object,2965,2.98,95664
order_estimated_delivery_date,object,0,0.00,459


### 3. Missing Value Investigation & Silver Layer Initialization

**Hypothesis:** Missing timestamps across the dataset are not systematic data loss, but rather represent orders currently in progress (e.g., in transit, canceled, or created but not yet shipped).

In [27]:
missing_any_status = (
    orders[orders.isna().any(axis=1)]["order_status"]
    .value_counts()
    .reset_index()
)

total_missing = missing_any_status["count"].sum()
print(f"Total rows with missing values: {total_missing}")

display(missing_any_status)

Total rows with missing values: 2980


,order_status,count
0,shipped,1107
1,canceled,619
2,unavailable,609
3,invoiced,314
4,processing,301
5,delivered,23
6,created,5
7,approved,2


#### Delivery Date Missingness Audit
Focusing specifically on `order_delivered_customer_date` to confirm whether missing dates align with undelivered order statuses.

In [28]:
missing_delivered_status = (
    orders[orders["order_delivered_customer_date"].isna()]["order_status"]
    .value_counts()
    .reset_index()
)

display(missing_delivered_status)

,order_status,count
0,shipped,1107
1,canceled,619
2,unavailable,609
3,invoiced,314
4,processing,301
5,delivered,8
6,created,5
7,approved,2


#### Edge Case Analysis: Isolating Anomalous Rows
Isolating edge cases where `order_delivered_customer_date` **is present**, but other critical timestamps are missing.

In [29]:
problematic_15 = orders[
    orders.isna().any(axis=1) & orders["order_delivered_customer_date"].notna()
]

missing_breakdown = problematic_15.isna().sum()
display(missing_breakdown[missing_breakdown > 0])

order_approved_at               14
order_delivered_carrier_date     1
dtype: int64

#### Initializing the Silver Layer
Filtering out the identified anomalous records to establish a clean and robust base dataset for further transformation.

In [30]:
orders_clean = orders[~orders.index.isin(problematic_15.index)].copy()

print(f"Original orders count:  {len(orders)}")
print(f"Cleaned orders count:   {len(orders_clean)}")

Original orders count:  99441
Cleaned orders count:   99426


### 4. Data Type Optimization

To transition our dataset from the Bronze layer to the Silver layer, we enforce robust data types:
- **Identifiers (`string`):** Standardize `order_id` and `customer_id` into explicit string types for schema consistency and relational integrity.
- **Categories (`category`):** Efficient for low-cardinality text columns (like `order_status`), reducing memory overhead and accelerating group-by operations.
- **Timestamps (`datetime`):** Essential for time-series analysis and calculating delivery delays.

In [31]:
orders_clean["order_id"] = orders_clean["order_id"].astype("string")
orders_clean["customer_id"] = orders_clean["customer_id"].astype("string")

orders_clean["order_status"] = orders_clean["order_status"].astype("category")

datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in datetime_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col])

display(orders_clean.dtypes.to_frame(name="Silver Layer Data Type"))

,Silver Layer Data Type
order_id,string[python]
customer_id,string[python]
order_status,category
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


### 5. Data Integrity Checks & Anomaly Detection
Ensuring logical consistency, validating business rules, and inspecting value distributions to isolate structural anomalies and outliers.

In [32]:
audit_checks = {
    "Approval before purchase":
        orders_clean["order_approved_at"] < orders_clean["order_purchase_timestamp"],
    "Carrier pickup before approval":
        orders_clean["order_delivered_carrier_date"] < orders_clean["order_approved_at"],
    "Customer delivery before carrier pickup":
        orders_clean["order_delivered_customer_date"] < orders_clean["order_delivered_carrier_date"],
    "Estimated delivery before purchase":
        orders_clean["order_estimated_delivery_date"] < orders_clean["order_purchase_timestamp"],
    "Delivered status, but missing delivery date":
        (orders_clean["order_status"] == "delivered") & (orders_clean["order_delivered_customer_date"].isna()),
    "Not delivered status, but has delivery date":
        (orders_clean["order_status"] != "delivered") & (orders_clean["order_delivered_customer_date"].notna()),
    "Canceled/unavailable status, but has carrier date":
        (orders_clean["order_status"].isin(["canceled", "unavailable"])) & (orders_clean["order_delivered_carrier_date"].notna()),
    "Delivered/shipped status, but missing approval date":
        (orders_clean["order_status"].isin(["delivered", "shipped"])) & (orders_clean["order_approved_at"].isna()),
    "Delivered/shipped status, but missing carrier date":
        (orders_clean["order_status"].isin(["delivered", "shipped"])) & (orders_clean["order_delivered_carrier_date"].isna()),
    "Missing core mandatory fields":
        orders_clean["order_purchase_timestamp"].isna() | orders_clean["customer_id"].isna() | orders_clean["order_status"].isna()
}

for label, condition in audit_checks.items():
    print(f"{label}: {condition.sum()}")

print(f"Duplicate order_id count: {orders_clean.duplicated(subset=['order_id']).sum()}")

Approval before purchase: 0
Carrier pickup before approval: 1359
Customer delivery before carrier pickup: 23
Estimated delivery before purchase: 0
Delivered status, but missing delivery date: 8
Not delivered status, but has delivery date: 6
Canceled/unavailable status, but has carrier date: 75
Delivered/shipped status, but missing approval date: 0
Delivered/shipped status, but missing carrier date: 1
Missing core mandatory fields: 0
Duplicate order_id count: 0


#### Delivery Duration Distribution Analysis

Isolate clean orders by filtering out logical anomalies and problematic rows, then calculate delivery lead times in days and inspect statistical quantiles to understand long-tail logistics patterns.

In [33]:
# Create the final audit mask from all logical checks
mask_audit = pd.DataFrame(audit_checks).any(axis=1)

# Update orders_clean by filtering out all logical anomalies permanently
orders_clean = orders_clean[~mask_audit].copy()

# Calculate delivery duration in days directly in orders_clean
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date'] - orders_clean['order_purchase_timestamp']
).dt.days

# Analyze delivery time distribution for successfully delivered orders
delivered_filter = orders_clean['order_status'] == 'delivered'
delivery_stats = orders_clean.loc[delivered_filter, 'delivery_days']

print("Delivery time statistics (Silver layer):")
display(delivery_stats.describe())

print("\nDelivery time quantiles:")
display(delivery_stats.quantile([0.90, 0.95, 0.99, 0.999]))

Delivery time statistics (Silver layer):


count    95082.000000
mean        12.152858
std          9.577214
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64


Delivery time quantiles:


0.900    23.0
0.950    29.0
0.990    46.0
0.999    83.0
Name: delivery_days, dtype: float64

#### Temporal Bounds Validation

Inspect the minimum and maximum dates across all core timestamp columns in the working dataset to verify that all records fall strictly within the realistic historical window of the Olist platform.

In [34]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

date_bounds = pd.DataFrame({
    'min_date': orders_clean[date_columns].min(),
    'max_date': orders_clean[date_columns].max()
})

display(date_bounds)

,min_date,max_date
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00


### 6. Silver Layer Validation & Export
Comparing Bronze vs. Silver metrics (shape, types, and record count) and persisting the dataset to Parquet.

In [35]:
print("Pipeline Summary: Bronze vs Silver Layer")
print(f"Original shape (Bronze): {orders.shape}")
print(f"Cleaned shape (Silver):  {orders_clean.shape}")
print(f"Total rows dropped:      {len(orders) - len(orders_clean):,}")

# Side-by-side comparison of data types
dtypes_comparison = pd.DataFrame({
    'Bronze (Raw)': orders.dtypes,
    'Silver (Clean)': orders_clean.dtypes
})

print("Data Types Transformation:")
display(dtypes_comparison)

Pipeline Summary: Bronze vs Silver Layer
Original shape (Bronze): (99441, 8)
Cleaned shape (Silver):  (97961, 9)
Total rows dropped:      1,480
Data Types Transformation:


,Bronze (Raw),Silver (Clean)
customer_id,object,string[python]
delivery_days,NaN,float64
order_approved_at,object,datetime64[ns]
order_delivered_carrier_date,object,datetime64[ns]
order_delivered_customer_date,object,datetime64[ns]
order_estimated_delivery_date,object,datetime64[ns]
order_id,object,string[python]
order_purchase_timestamp,object,datetime64[ns]
order_status,object,category


### 7. Data Quality Assertions & Export

Before persisting the clean dataset to the Silver layer, we execute automated data quality checks to validate logical integrity, schema consistency, and business rules:

* **Primary Key Integrity:** Ensure `order_id` contains no duplicate or null values.
* **Mandatory Attributes:** Verify critical fields (`customer_id`, `order_status`, `order_purchase_timestamp`) are non-null.
* **Logical Sequence:** Confirm delivery dates do not precede purchase timestamps.
* **Dataset Validity:** Ensure the resulting dataset is non-empty before writing to storage.

In [36]:
assert orders_clean['order_id'].is_unique, "Assertion failed: Duplicate order_id found"
assert orders_clean['order_id'].notna().all(), "Assertion failed: NULL values found in order_id"

critical_cols = ['customer_id', 'order_status', 'order_purchase_timestamp']
for col in critical_cols:
    assert orders_clean[col].notna().all(), f"Assertion failed: NULL values found in mandatory column '{col}'"

invalid_delivery = (
    orders_clean['order_delivered_customer_date'].notna() & 
    (orders_clean['order_delivered_customer_date'] < orders_clean['order_purchase_timestamp'])
)
assert not invalid_delivery.any(), "Assertion failed: Delivery date is earlier than purchase date"

assert len(orders_clean) > 0, "Assertion failed: Cleaned dataset is empty"

print("All quality checks passed successfully. Saving to Silver layer.")
orders_clean.to_parquet("../data/orders_clean.parquet", index=False)
print("Orders dataset successfully saved to Parquet.")

All quality checks passed successfully. Saving to Silver layer.


Orders dataset successfully saved to Parquet.
